<a href="https://colab.research.google.com/github/daniausman24-bot/ML_Internship_Track/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daniausman24-bot/ML_Internship_Track/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/daniausman24-bot/ML_Internship_Track"
REPO_DIR = "ML_Internship_Track"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/ML_Internship_Track
Starter data found. You're ready.


## 1. Method choice and why

I will start with Logistic Regression because the target is binary: declining versus not declining. The model gives a probability of decline, which can be used to rank content for review.

Logistic Regression is also easier to interpret than a more complex model, so it gives a useful first comparison against the Week-4 rule baseline.

If the learned model provides a useful improvement over the baseline, a stronger model can be considered later. The goal is not to add complexity unless the comparison shows a benefit.

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

# Reproducibility
RANDOM_STATE = 42

# Find the starter data
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in possible_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError("Starter CSV was not found.")

df = pd.read_csv(data_path)

# Binary target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Data shape:", df.shape)
print("Declining rate:", df["is_declining_label"].mean())
print("Random state:", RANDOM_STATE)

Data shape: (30000, 45)
Declining rate: 0.5420666666666667
Random state: 42


## 2. Split design

I will use a grouped train/test split by client_id.

The split keeps complete clients in either the training set or the test set rather than putting rows from the same client into both sets. This reduces the chance that client-specific patterns leak between training and testing.

I will use 80% of the clients for training and 20% for testing, with a fixed random seed of 42 for reproducibility.

The test set will be used only for the final comparison between the learned model and the Week-4 baseline.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(df, df["is_declining_label"], groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTraining clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print("\nClient overlap:")
print(
    len(
        set(train_df["client_id"])
        & set(test_df["client_id"])
    )
)

print("\nTraining declining rate:",
      train_df["is_declining_label"].mean())

print("Test declining rate:",
      test_df["is_declining_label"].mean())

Training rows: 23837
Test rows: 6163

Training clients: 25
Test clients: 7

Client overlap:
0

Training declining rate: 0.5501111717078492
Test declining rate: 0.5109524582184002


## 3. Train + compare vs my baseline

The Logistic Regression model was trained on 23,837 rows from 25 clients and evaluated on 6,163 rows from 7 different clients. There was zero client overlap between training and testing.

On the held-out test set, the Logistic Regression model achieved 0.70 Precision@20 and 0.72 Precision@50. The Week-4 baseline achieved 0.35 Precision@20 and 0.46 Precision@50.

The test-set declining rate was 0.511. Therefore, the Logistic Regression model's Precision@20 was about 18.9 percentage points above the test base rate, while the Week-4 baseline was below the base rate at 0.35.

Method	Precision@20	Precision@50	Test base rate
Week-4 baseline	0.35	0.46	0.511
Logistic Regression	0.70	0.72	0.511

The learned model performed better than the simple rule on both ranking metrics on this held-out grouped test set. This result is evidence of useful ranking signal, not proof that the model will generalize equally well to every future client.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Raw numeric features
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

# Create log versions of skewed traffic variables
for frame in [train_df, test_df]:
    frame["log_impressions_90d"] = np.log1p(frame["impressions_90d"])
    frame["log_clicks_90d"] = np.log1p(frame["clicks_90d"])
    frame["log_sessions_90d"] = np.log1p(frame["sessions_90d"])
    frame["log_ai_sessions_90d"] = np.log1p(frame["ai_sessions_90d"])

model_numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

X_train = train_df[model_numeric_features + categorical_features]
X_test = test_df[model_numeric_features + categorical_features]

y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, model_numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)

model_test_probability = model.predict_proba(X_test)[:, 1]

print("Model trained.")
print("Test predictions:", len(model_test_probability))

Model trained.
Test predictions: 6163


In [ ]:
# -----------------------------
# Week-4 baseline on TEST ONLY
# -----------------------------

test_baseline = test_df.copy()

test_baseline["stale_flag"] = (
    test_baseline["days_since_last_update"] >= 180
).astype(int)

test_baseline["visible_flag"] = (
    test_baseline["impressions_90d"] >= 500
).astype(int)

test_baseline["weak_position_flag"] = (
    (test_baseline["avg_position"] > 0) &
    (test_baseline["avg_position"] >= 10)
).astype(int)

test_baseline["baseline_score"] = (
    test_baseline["stale_flag"]
    + test_baseline["visible_flag"]
    + test_baseline["weak_position_flag"]
)

In [ ]:
def precision_at_k(y_true, scores, k):
    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top = temp.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top["y"].mean()


# Model scores
model_p20 = precision_at_k(
    y_test,
    model_test_probability,
    20
)

model_p50 = precision_at_k(
    y_test,
    model_test_probability,
    50
)

# Baseline scores
baseline_p20 = precision_at_k(
    y_test,
    test_baseline["baseline_score"],
    20
)

baseline_p50 = precision_at_k(
    y_test,
    test_baseline["baseline_score"],
    50
)

base_rate = y_test.mean()

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        baseline_p20,
        model_p20
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ],
    "Base rate": [
        base_rate,
        base_rate
    ]
})

print(comparison.to_string(index=False))

             Method  Precision@20  Precision@50  Base rate
    Week-4 baseline          0.35          0.46   0.510952
Logistic Regression          0.70          0.72   0.510952


In [ ]:
# Get transformed feature names
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

coefficients = (
    model.named_steps["classifier"].coef_[0]
)

importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
}).sort_values(
    "abs_coefficient",
    ascending=False
)

print("Top 15 features by absolute coefficient:")
print(
    importance.head(15)[
        ["feature", "coefficient"]
    ].to_string(index=False)
)

Top 15 features by absolute coefficient:
                                     feature  coefficient
                numeric__log_impressions_90d     1.443805
            categorical__position_tier_top_3    -0.781745
            categorical__impression_tier_low     0.774233
      categorical__impression_tier_excellent    -0.595214
                     numeric__log_clicks_90d    -0.562740
      categorical__word_count_tier_1000-2000     0.554922
categorical__content_type_comparison article    -0.510888
           categorical__freshness_tier_31-90    -0.437708
                       numeric__avg_position    -0.380992
                         numeric__word_count     0.358882
      categorical__word_count_tier_2000-3500    -0.344213
                   numeric__content_age_days    -0.338775
   categorical__content_type_keyword article     0.326898
                   numeric__log_sessions_90d    -0.315266
          categorical__word_count_tier_3500+    -0.297626


## 4. Errors and interpretation

The Logistic Regression model made 2,559 errors out of 6,163 test rows, giving an error rate of 0.415. The errors include both false positives and false negatives.

The false-positive examples received high predicted probabilities even though their observed label was not declining. The three examples shown by the error check had predicted probabilities between 0.932 and 0.955. They were keyword articles, and their observed impressions and positions varied. This shows that a high model score does not guarantee that a page is actually declining.

The false-negative examples had very low observed impressions, between 2 and 3 impressions over 90 days. Their predicted probabilities were also very low, between 0.064 and 0.080, even though their observed label was declining. This suggests that the model can miss some low-traffic declining content because there is limited traffic signal available.

The test set contained only keyword articles, so the error-rate-by-content-type check does not provide a meaningful comparison between content types.

The strongest model coefficients were associated with log 90-day impressions, position tier, impression tier, log clicks, word-count tier, content type, freshness tier, average position, word count, and content age. These coefficients show associations learned by the model and should not be interpreted as causal effects.

Overall, Logistic Regression performed better than the Week-4 baseline on the same grouped test set. Its Precision@20 was 0.70 compared with 0.35 for the baseline, while Precision@50 was 0.72 compared with 0.46. The model therefore provides stronger ranking performance in this test, but its errors show that low-traffic content remains difficult to identify reliably.


In [ ]:
print("Total test errors:", error_df["error"].sum())
print("Test error rate:", error_df["error"].mean())

print("\nFalse positives:")
print(
    error_df[
        (error_df["predicted_label"] == 1) &
        (error_df["is_declining_label"] == 0)
    ]
    .sort_values("predicted_probability", ascending=False)
    .head(3)
    .to_string(index=False)
)

print("\nFalse negatives:")
print(
    error_df[
        (error_df["predicted_label"] == 0) &
        (error_df["is_declining_label"] == 1)
    ]
    .sort_values("predicted_probability", ascending=True)
    .head(3)
    .to_string(index=False)
)

Total test errors: 2559
Test error rate: 0.41521986045756937

False positives:
          content_id         client_id  is_declining_label  impressions_90d  days_since_last_update  avg_position    content_type   main_intent  predicted_probability  predicted_label  error
content_7be5f150dc65 client_f369cb89fc                   0              290                      20           5.9 keyword article informational               0.954821                1   True
content_41baf0722ad9 client_8527a891e2                   0             3115                     104          12.8 keyword article informational               0.943097                1   True
content_5d5653c4eb4f client_4e07408562                   0            15101                       7           5.7 keyword article informational               0.932140                1   True

False negatives:
          content_id         client_id  is_declining_label  impressions_90d  days_since_last_update  avg_position    content_type   main_in

In [ ]:
print("\nError rate by content type:")

print(
    error_df.groupby("content_type")["error"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)


Error rate by content type:
                 count     mean
content_type                   
keyword article   6163  0.41522


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.